## MODEL A ( All Features )

### 1. Import the libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import joblib


### 2. Load the dataset

In [3]:
data = pd.read_csv(r"C:\Users\HP\Documents\PycharmProjects\network-anomaly-detector\Detector\data\processed-data\clean-data.csv", low_memory=False)
print(data.shape)
data.sample(10)

(2540047, 45)


,sport,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,service,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
2449076,47439.0,53.0,udp,INT,0.000004,114,0,254,0,dns,...,0,33,33,16,17,16,16,34,Generic,1
1838440,35430.0,53.0,udp,CON,0.000988,130,162,31,29,dns,...,0,3,3,2,4,2,1,3,Normal,0
2377679,33927.0,143.0,tcp,FIN,2.039919,7814,15134,31,29,unknown,...,0,8,4,1,5,2,1,1,Normal,0
1950680,14771.0,13637.0,tcp,FIN,0.363294,4976,3080,31,29,unknown,...,0,14,5,6,12,1,1,4,Normal,0
1065837,33314.0,31833.0,tcp,FIN,0.376938,4776,3080,31,29,unknown,...,0,7,5,5,6,1,1,3,Normal,0
1474580,19123.0,16826.0,tcp,FIN,0.033186,3078,31400,31,29,unknown,...,0,6,3,2,2,1,1,2,Normal,0
292505,2039.0,31198.0,tcp,FIN,0.013193,2646,24004,31,29,unknown,...,0,4,7,2,3,1,1,1,Normal,0
479854,52632.0,53319.0,tcp,FIN,0.006334,4040,2456,31,29,unknown,...,0,19,16,12,17,4,4,7,Normal,0
1946376,6401.0,53.0,udp,CON,0.001079,146,178,31,29,dns,...,0,3,1,2,4,2,1,1,Normal,0
1481779,11806.0,111.0,udp,CON,0.004396,568,304,31,29,unknown,...,0,8,5,7,5,1,1,3,Normal,0


### 3. Train-Test-Split

In [4]:
X = data.drop(columns=["Label", "attack_cat"])
y = data["Label"]
y_cat = data["attack_cat"]

X_train, X_test, y_train, y_test, y_cat_train, y_cat_test = train_test_split(
    X,
    y,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(y_cat_train.shape)
print(y_cat_test.shape)

(2032037, 43)
(508010, 43)
(2032037,)
(508010,)
(2032037,)
(508010,)


### 4. Separate the diffent type of features

In [5]:
numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns
categorical_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns

print("Numeric Features :\n",numeric_cols)
print("\nCategorical Features:\n",categorical_cols)

Numeric Features :
 Index(['sport', 'dsport', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'Sload',
       'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime',
       'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm'],
      dtype='str')

Categorical Features:
 Index(['proto', 'state', 'service'], dtype='str')


Making custom transformer for grouping low frequency categories

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategories(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=7):
        self.top_n = top_n
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            self.top_categories_[col] = (
                X[col].value_counts()
                .head(self.top_n)
                .index
                .tolist()
            )
        return self

    def transform(self, X):
        X = X.copy()

        for col in X.columns:
            X[col] = X[col].where(
                X[col].isin(self.top_categories_[col]),
                "other"
            )

        return X
    def get_feature_names_out(self, input_features=None):
        return input_features

### 5. Creating pipelines for Feature Engineering

In [7]:
num_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent').set_output(transform = 'pandas')),
    ('Grouping', RareCategories(top_n=7)),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])
label_enc = LabelEncoder()

### 6. Using Column Transformer

In [8]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_cols),
    ("cat", cat_pipeline, categorical_cols),
])

#### *** Final Dataset ***

In [9]:
X_train_t = preprocessor.fit_transform(X_train)

print(X_train_t.shape)

X_train_prev = pd.DataFrame(
    X_train_t[:5],
    columns=preprocessor.get_feature_names_out()
)

pd.set_option('display.max_columns', None)
X_train_prev

(2032037, 64)


,num__sport,num__dsport,num__dur,num__sbytes,num__dbytes,num__sttl,num__dttl,num__Sload,num__Dload,num__Spkts,num__Dpkts,num__swin,num__dwin,num__stcpb,num__dtcpb,num__smeansz,num__dmeansz,num__trans_depth,num__res_bdy_len,num__Sjit,num__Djit,num__Stime,num__Ltime,num__Sintpkt,num__Dintpkt,num__tcprtt,num__synack,num__ackdat,num__is_sm_ips_ports,num__ct_state_ttl,num__ct_flw_http_mthd,num__is_ftp_login,num__ct_ftp_cmd,num__ct_srv_src,num__ct_srv_dst,num__ct_dst_ltm,num__ct_src_ ltm,num__ct_src_dport_ltm,num__ct_dst_sport_ltm,num__ct_dst_src_ltm,cat__proto_arp,cat__proto_icmp,cat__proto_ospf,cat__proto_other,cat__proto_sctp,cat__proto_tcp,cat__proto_udp,cat__proto_unas,cat__state_CLO,cat__state_CON,cat__state_ECO,cat__state_FIN,cat__state_INT,cat__state_REQ,cat__state_RST,cat__state_other,cat__service_dns,cat__service_ftp,cat__service_ftp-data,cat__service_http,cat__service_other,cat__service_smtp,cat__service_ssh,cat__service_unknown
0,1.640403,1.403386,-0.048211,0.010891,0.273297,-0.425809,-0.041389,-0.302995,3.413645,0.681621,0.356940,0.835793,0.838102,0.824463,-0.686602,-0.436201,1.959374,-0.237160,-0.089600,-0.092817,-0.208815,-1.139694,-1.139695,-0.069347,-0.054780,-0.116231,-0.102046,-0.113058,-0.040411,-0.382176,-0.198043,-0.12989,-0.111546,-0.665216,0.093551,-0.666453,-0.597413,-0.429634,-0.419922,-0.519295,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-1.069618,-0.604947,0.460503,0.282126,6.530936,-0.425809,-0.041389,-0.311417,-0.265935,4.333078,5.793729,0.835793,0.838102,1.100049,-0.407558,-0.469133,3.521227,2.613055,10.998107,0.089459,0.411228,0.855404,0.855410,-0.062832,-0.048975,-0.118820,-0.104653,-0.115215,-0.040411,-0.382176,1.600785,-0.12989,-0.111546,-0.757534,-0.738433,-0.421413,-0.719279,-0.429634,-0.419922,-0.519295,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.826986,-0.606411,-0.051153,-0.077358,-0.225849,-0.037138,-0.718072,1.917087,-0.580104,-0.419186,-0.351490,-1.196484,-1.193180,-0.887568,-0.887493,0.051192,-0.824540,-0.237160,-0.089600,-0.094211,-0.215482,-1.170979,-1.170980,-0.069507,-0.055082,-0.133490,-0.126123,-0.120192,-0.040411,-0.382176,-0.198043,-0.12989,-0.111546,-0.295941,-0.276219,-0.421413,-0.475546,-0.193688,-0.095887,-0.341567,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.385067,-0.604947,0.045548,-0.050104,-0.162696,-0.425809,-0.041389,-0.311527,-0.565409,-0.258093,-0.203214,0.835793,0.838102,-0.134019,1.424864,-0.027845,0.859519,2.613055,-0.006615,0.435577,2.092206,-1.148598,-1.148598,-0.035154,-0.004523,-0.117202,-0.102468,-0.114468,-0.040411,-0.382176,1.600785,-0.12989,-0.111546,-0.665216,-0.553547,-0.543933,-0.353680,-0.311661,-0.419922,-0.519295,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-1.493940,-0.609285,-0.051153,-0.078587,-0.225849,2.562940,-0.718072,0.653193,-0.580104,-0.419186,-0.351490,-1.196484,-1.193180,-0.887568,-0.887493,-0.159573,-0.824540,-0.237160,-0.089600,-0.094211,-0.215482,0.873844,0.873844,-0.069506,-0.055082,-0.133490,-0.126123,-0.120192,-0.040411,2.546543,-0.198043,-0.12989,-0.111546,-0.480578,-0.461105,-0.543933,-0.597413,-0.311661,-0.257905,-0.252703,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### 7. Creating pipelines for models

In [62]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators = 100,max_depth = 15, random_state = 42))
])
rf_cat_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators = 100, max_depth = 15, random_state = 42))
])

Applying LabelEncoder on attack_cat

In [11]:
y_cat_train_enc = label_enc.fit_transform(y_cat_train)
y_cat_test_enc = label_enc.transform(y_cat_test)

print(dict(zip(
    label_enc.classes_,
    label_enc.transform(label_enc.classes_)
)))

{'Analysis': np.int64(0), 'Backdoor': np.int64(1), 'DoS': np.int64(2), 'Exploits': np.int64(3), 'Fuzzers': np.int64(4), 'Generic': np.int64(5), 'Normal': np.int64(6), 'Reconnaissance': np.int64(7), 'Shellcode': np.int64(8), 'Worms': np.int64(9)}


### 8. Model training

In [63]:
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
rf_cat_pipeline.fit(X_train, y_cat_train_enc)

lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)
rf_cat_pred = rf_cat_pipeline.predict(X_test)

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 9. Model Evaluation

In [64]:
print("Logistic Regression")
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall   :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))

print("Classification Report:")
print(classification_report(y_test, lr_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))


print("\nRandom Forest")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))

print("Classification Report:")
print(classification_report(y_test, rf_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))


print("\nRandom Forest - Attack Category")
print("Accuracy :", accuracy_score(y_cat_test_enc, rf_cat_pred))
print("Precision:", precision_score(y_cat_test_enc, rf_cat_pred,  zero_division=0,average="weighted"))
print("Recall   :", recall_score(y_cat_test_enc, rf_cat_pred, average="weighted"))
print("F1 Score :", f1_score(y_cat_test_enc, rf_cat_pred, average="weighted"))

print("Classification Report:")
print(classification_report(y_cat_test_enc, rf_cat_pred, zero_division=0, target_names=label_enc.classes_))

print("Confusion Matrix:")
print(confusion_matrix(y_cat_test_enc, rf_cat_pred))

Logistic Regression
Accuracy : 0.9895828822267279
Precision: 0.946413700165044
Recall   : 0.9727189255645299
F1 Score : 0.9593860322333078
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    443753
           1       0.95      0.97      0.96     64257

    accuracy                           0.99    508010
   macro avg       0.97      0.98      0.98    508010
weighted avg       0.99      0.99      0.99    508010

Confusion Matrix:
[[440214   3539]
 [  1753  62504]]

Random Forest
Accuracy : 0.9948878959075609
Precision: 0.9839418579097731
Recall   : 0.9755046142832687
F1 Score : 0.9797050709970851
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    443753
           1       0.98      0.98      0.98     64257

    accuracy                           0.99    508010
   macro avg       0.99      0.99      0.99    508010
weighted avg       0.99     

### 10. Overfitting Check

In [65]:
rf_train_pred = rf_pipeline.predict(X_train)
rf_test_pred = rf_pipeline.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, rf_train_pred))
print("Test Accuracy :", accuracy_score(y_test, rf_test_pred))

print("\nTrain F1:", f1_score(y_train, rf_train_pred))
print("Test F1 :", f1_score(y_test, rf_test_pred))


rf_cat_train_pred = rf_cat_pipeline.predict(X_train)
rf_cat_test_pred = rf_cat_pipeline.predict(X_test)

rf_cat_train_pred = label_enc.inverse_transform(rf_cat_train_pred)
rf_cat_test_pred = label_enc.inverse_transform(rf_cat_test_pred)

print("\nCategory Train Accuracy:", accuracy_score(y_cat_train, rf_cat_train_pred))
print("Category Test Accuracy :", accuracy_score(y_cat_test, rf_cat_test_pred))

print("\nCategory Train F1:", f1_score(y_cat_train, rf_cat_train_pred, average="weighted"))
print("Category Test F1 :", f1_score(y_cat_test, rf_cat_test_pred, average="weighted"))

Train Accuracy: 0.9959656246416773
Test Accuracy : 0.9948878959075609

Train F1: 0.9839662348228815
Test F1 : 0.9797050709970851

Category Train Accuracy: 0.9846385671127051
Category Test Accuracy : 0.9818842148776599

Category Train F1: 0.9832565917499686
Category Test F1 : 0.9803095833774175


### 11. Prediction

In [66]:
sample = X_test.head(10)
print(y_test.head(10))
print(y_cat_test.head(10))
binary_pred = rf_pipeline.predict(sample)
category_pred = rf_cat_pipeline.predict(sample)

category_pred = label_enc.inverse_transform(category_pred)

result = pd.DataFrame({
    "Attack Prediction": binary_pred,
    "Category Prediction": category_pred
})
result

1534731    0
1502515    0
1870008    1
1588250    0
2481022    0
2197385    0
1934080    0
290940     0
209730     0
1148719    1
Name: Label, dtype: int64
1534731      Normal
1502515      Normal
1870008     Generic
1588250      Normal
2481022      Normal
2197385      Normal
1934080      Normal
290940       Normal
209730       Normal
1148719    Exploits
Name: attack_cat, dtype: str


,Attack Prediction,Category Prediction
0,0,Normal
1,0,Normal
2,1,Exploits
3,0,Normal
4,0,Normal
5,0,Normal
6,0,Normal
7,0,Normal
8,0,Normal
9,1,DoS


### Saving the model as onnx file

In [52]:
import onnx
from onnx import helper, TensorProto
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from pathlib import Path

# --------------------------------------------------
# SETUP DIRECTORY
# --------------------------------------------------
model_dir = Path(
    r"C:\Users\HP\Documents\PycharmProjects\network-anomaly-detector\Detector\models"
)
model_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# EXTRACT CLASSIFIERS
# --------------------------------------------------
binary_model = rf_pipeline.named_steps["classifier"]
category_model = rf_cat_pipeline.named_steps["classifier"]

# --------------------------------------------------
# DEFINE INPUT AND OPTIONS
# --------------------------------------------------
initial_type = [("X", FloatTensorType([None, 64]))]

binary_options = {id(binary_model): {'zipmap': False}}
category_options = {id(category_model): {'zipmap': False}}

target_opsets = {'ai.onnx.ml': 3, '': 14}

# --------------------------------------------------
# CONVERT MODELS TO ONNX
# --------------------------------------------------
binary_onnx = convert_sklearn(
    binary_model,
    initial_types=initial_type,
    options=binary_options,
    target_opset=target_opsets
)

category_onnx = convert_sklearn(
    category_model,
    initial_types=initial_type,
    options=category_options,
    target_opset=target_opsets
)

# --------------------------------------------------
# RENAME INTERNAL NODES, OUTPUTS, AND INITIALIZERS
# --------------------------------------------------
initializers = []

for idx, node in enumerate(binary_onnx.graph.node):
    node.name = f"binary_{node.name}" if node.name else f"binary_node_{idx}"
    for i in range(len(node.input)):
        if node.input[i] != "X":
            node.input[i] = "binary_" + node.input[i]
    for i in range(len(node.output)):
        node.output[i] = "binary_" + node.output[i]

for init in binary_onnx.graph.initializer:
    init.name = "binary_" + init.name
    initializers.append(init)

for idx, node in enumerate(category_onnx.graph.node):
    node.name = f"category_{node.name}" if node.name else f"category_node_{idx}"
    for i in range(len(node.input)):
        if node.input[i] != "X":
            node.input[i] = "category_" + node.input[i]
    for i in range(len(node.output)):
        node.output[i] = "category_" + node.output[i]

for init in category_onnx.graph.initializer:
    init.name = "category_" + init.name
    initializers.append(init)

# --------------------------------------------------
# GET PREDICTION OUTPUTS
# --------------------------------------------------
binary_pred = "binary_" + binary_onnx.graph.output[0].name
category_pred = "category_" + category_onnx.graph.output[0].name

# --------------------------------------------------
# CREATE ONE GRAPH
# --------------------------------------------------
nodes = []
nodes.extend(binary_onnx.graph.node)
nodes.extend(category_onnx.graph.node)

binary_cast_output = "binary_pred_float"
nodes.append(
    helper.make_node(
        "Cast",
        name="binary_cast_node",
        inputs=[binary_pred],
        outputs=[binary_cast_output],
        to=TensorProto.FLOAT,
    )
)

category_cast_output = "category_pred_float"
nodes.append(
    helper.make_node(
        "Cast",
        name="category_cast_node",
        inputs=[category_pred],
        outputs=[category_cast_output],
        to=TensorProto.FLOAT,
    )
)

# --------------------------------------------------
# ADD RESHAPE NODES TO FIX CONCAT DIMENSIONS
# --------------------------------------------------
shape_tensor = helper.make_tensor(
    name="reshape_dim",
    data_type=TensorProto.INT64,
    dims=[2],
    vals=[-1, 1]
)
initializers.append(shape_tensor)

binary_reshaped = "binary_pred_reshaped"
nodes.append(
    helper.make_node(
        "Reshape",
        name="binary_reshape_node",
        inputs=[binary_cast_output, "reshape_dim"],
        outputs=[binary_reshaped]
    )
)

category_reshaped = "category_pred_reshaped"
nodes.append(
    helper.make_node(
        "Reshape",
        name="category_reshape_node",
        inputs=[category_cast_output, "reshape_dim"],
        outputs=[category_reshaped]
    )
)

# --------------------------------------------------
# CONCATENATE THE RESHAPED OUTPUTS
# --------------------------------------------------
concat_node = helper.make_node(
        "Concat",
        name="prediction_concat_node",
        inputs=[binary_reshaped, category_reshaped],
        outputs=["predictions"],
        axis=1,
)
nodes.append(concat_node)

# --------------------------------------------------
# BUILD THE FINAL GRAPH
# --------------------------------------------------
graph = helper.make_graph(
    nodes,
    "NetworkAnomalyDetector",
    [
        helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, 64])
    ],
    [
        helper.make_tensor_value_info("predictions", TensorProto.FLOAT, [None, 2])
    ],
    initializer=initializers
)

# --------------------------------------------------
# CREATE AND SAVE MODEL
# --------------------------------------------------
combined_onnx = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 14),
        helper.make_opsetid("ai.onnx.ml", 3)
    ]
)

onnx_path = model_dir / "network_anomaly_detector.onnx"
onnx.save(combined_onnx, onnx_path)

print("ONNX model saved successfully:", onnx_path)

ONNX model saved successfully: C:\Users\HP\Documents\PycharmProjects\network-anomaly-detector\Detector\models\network_anomaly_detector.onnx
